# Jitter engine pipeline demonstration
This notebook demonstrate how to use the full jitter pipeline.  The main engine is contained in the `JitterEvaluator` class of the `jitter` module.  The `JitterEvaluator` class is fundamentally a wrapper of three models:
- a contextual embedder from HuggingFace,
- a logistic regression for relevance filtering,
- a logistic regression for jitter scoring.

In [1]:
import pandas as pd
import ast
from jitter import JitterEvaluator, get_headlines

The engine can be instantiated by passing a string describing the HF model.  If no string is passed, it defaults to `ProsusAI/finbert`.

In [2]:
engine = JitterEvaluator()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The engine object needs to be trained on a training dataset.  The training dataset must contain (at least) the following four columns:
- `embedding`: the embedded sentence,
- `relevant`: 0 if not relevant, 1 if relevant,
- `jittery`: 0 if not jittery, 1 if jittery.

At startup, we load a pre-labeled dataset from `training.csv`.  Because CSV cannot hold list types, we need to evaluate the `embedding` string column to a list.

In [3]:
training = pd.read_csv("training.csv")
training["embedding"] = training.embedding.apply(ast.literal_eval)

We can then train the engine with the training dataset.

In [4]:
engine.train(training)

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
/Users/

Next, we need some headlines to score.  We use the `get_headlines` function from the `jitter` module.  The function takes a list of RSS urls and an optional date parameter, and returns the headlines from the given urls on the given date as `pd.Series`.  The optional `date` parameter is a string of format `YYYY-MM-DD`, but can also be set to `today` to fetch today's headlines only.

In [5]:
urls = pd.read_csv("sources.csv").url
headlines = get_headlines(urls, date="today")

Once we have the headlines, we process them.  This means that the engine populates an internal dataframe with the headlines, computes their embeddings, estimates which ones are relevant, and predicts which of the relevant headlines are jittery or not.

In [6]:
engine.process_headlines(headlines)

The `total_jitter` property of engine gives the fraction of relevant headlines that are labeled as "jittery", and thus provides an estimate of the current jitteriness.

In [7]:
engine.total_jitter

array([0.61714286])